# Week 2 — Sentiment Analysis (VADER Pipeline)

This notebook covers the first half of Week 2's sentiment analysis pipeline for the Jindal Steel Limited B2B Quality Feedback dataset. We implement a rule-based sentiment classifier using NLTK's VADER (Valence Aware Dictionary and sEntiment Reasoner) SentimentIntensityAnalyzer. VADER evaluates the sentiment polarity of the raw review text and categorizes each feedback entry as positive, negative, or neutral. Finally, we derive a ground-truth sentiment label from the severity ratings and perform a distribution sanity check.


In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer


In [2]:
# Download the VADER lexicon package from NLTK
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/faizali1/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [3]:
# Load cleaned dataset from Week 1
cleaned_path = '../data/processed/reviews_cleaned.csv'
df = pd.read_csv(cleaned_path)

print("Cleaned dataset shape:", df.shape)
df.head()

Cleaned dataset shape: (10000, 14)


,feedback_id,feedback_date,customer_type,customer_region,product_category,order_quantity_mt,severity_rating,feedback_text,cleaned_text,tokens,feedback_topic,feedback_source,plant_location,resolution_status
0,JSL-00001,2022-10-16,Construction Company,South India,Wire Rods,36,3,Our 36 MT Wire Rods shipment from Patratu plan...,our 36 mt wire rods shipment from patratu plan...,"['36', 'mt', 'wire', 'rod', 'shipment', 'patra...",packaging_handling,Key Account Manager Visit,Patratu (Jharkhand),Escalated
1,JSL-00002,2022-11-11,Construction Company,South India,TMT Rebars (Jindal Panther),479,3,We received the 479 MT batch of Jindal Panther...,we received the 479 mt batch of jindal panther...,"['received', '479', 'mt', 'batch', 'jindal', '...",customer_service,Email Submission,Angul (Odisha),Resolved
2,JSL-00003,2024-03-06,Construction Company,North India,Beams and Columns (PF Sections),25,4,The 25 MT consignment of PF sections received ...,the 25 mt consignment of pf sections received ...,"['25', 'mt', 'consignment', 'pf', 'section', '...",product_quality,Key Account Manager Visit,Angul (Odisha),Resolved
3,JSL-00004,2022-10-23,Real Estate Developer,West India,Jindal Panther Cement,744,4,We have been sourcing Jindal Panther Cement fr...,we have been sourcing jindal panther cement fr...,"['sourcing', 'jindal', 'panther', 'cement', 'p...",pricing_commercial,Key Account Manager Visit,Patratu (Jharkhand),Resolved
4,JSL-00005,2024-12-15,Construction Company,North India,Jindal Speedfloor,433,1,Our construction division placed a 433 MT orde...,our construction division placed a 433 mt orde...,"['construction', 'division', 'placed', '433', ...",customer_service,Dealer Survey Form,Angul (Odisha),Pending


In [4]:
# Function to convert VADER compound score into categorical labels
def get_vader_label(compound):
    if compound >= 0.05:
        return 'positive'
    elif compound <= -0.05:
        return 'negative'
    else:
        return 'neutral'

# Function to derive ground-truth sentiment labels from severity rating
def get_rating_based_truth(rating):
    if rating in [1, 2]:
        return 'negative'
    elif rating == 3:
        return 'neutral'
    elif rating in [4, 5]:
        return 'positive'
    else:
        return 'unknown'


In [5]:
# Initialize VADER Sentiment Intensity Analyzer
sia = SentimentIntensityAnalyzer()

# Apply VADER polarity scores to the RAW uncleaned review text (feedback_text)
df['vader_scores'] = df['feedback_text'].apply(lambda x: sia.polarity_scores(str(x)))
df['vader_compound'] = df['vader_scores'].apply(lambda x: x['compound'])

# Assign categorical labels based on compound scores
df['vader_label'] = df['vader_compound'].apply(get_vader_label)

# Derive ground-truth labels from severity ratings
df['rating_based_truth'] = df['severity_rating'].apply(get_rating_based_truth)

# Select and rename columns as requested: review text, rating, vader_label, rating_based_truth
sentiment_df = df.copy()
sentiment_df = sentiment_df.rename(columns={
    'feedback_text': 'review text',
    'severity_rating': 'rating'
})
output_df = sentiment_df[['review text', 'rating', 'vader_label', 'rating_based_truth']]
output_df.head(10)

,review text,rating,vader_label,rating_based_truth
0,Our 36 MT Wire Rods shipment from Patratu plan...,3,positive,neutral
1,We received the 479 MT batch of Jindal Panther...,3,positive,neutral
2,The 25 MT consignment of PF sections received ...,4,positive,positive
3,We have been sourcing Jindal Panther Cement fr...,4,positive,positive
4,Our construction division placed a 433 MT orde...,1,positive,negative
5,The 491 MT dispatch lot of Steel Rails from Ba...,5,positive,positive
6,Our team handling the Char Dham highway projec...,5,positive,positive
7,We received 441 MT of Wire Rods from Raigarh (...,3,positive,neutral
8,The billets and blooms we procured from Patrat...,5,positive,positive
9,The 512 MT consignment of Wire Rods dispatched...,4,negative,positive


In [6]:
# Save the intermediate dataset to outputs/sentiment_stage1.csv
output_path = '../outputs/sentiment_stage1.csv'
output_df.to_csv(output_path, index=False)

print(f"Saved intermediate sentiment dataset to: {output_path}")
print("Shape:", output_df.shape)


Saved intermediate sentiment dataset to: ../outputs/sentiment_stage1.csv
Shape: (10000, 4)


In [7]:
# Print distribution percentages for both columns
print("VADER Class Distribution (%):")
print(output_df['vader_label'].value_counts(normalize=True) * 100)
print("-" * 50)
print("Rating-Based Ground Truth Distribution (%):")
print(output_df['rating_based_truth'].value_counts(normalize=True) * 100)
print("-" * 50)

# Cross tabulation of VADER vs Rating-Based Truth
print("Cross-Tabulation of Truth vs. VADER Predictions:")
print(pd.crosstab(output_df['rating_based_truth'], output_df['vader_label'], margins=True))


VADER Class Distribution (%):
vader_label
positive    73.15
negative    22.95
neutral      3.90
Name: proportion, dtype: float64
--------------------------------------------------
Rating-Based Ground Truth Distribution (%):
rating_based_truth
positive    55.22
negative    24.98
neutral     19.80
Name: proportion, dtype: float64
--------------------------------------------------
Cross-Tabulation of Truth vs. VADER Predictions:
vader_label         negative  neutral  positive    All
rating_based_truth                                    
negative                1660      113       725   2498
neutral                  275       92      1613   1980
positive                 360      185      4977   5522
All                     2295      390      7315  10000


### VADER vs. Rating-Based Sentiment Sanity Check

A quick review of the sentiment distribution highlights key differences between VADER's rule-based predictions and the rating-based ground truth:

1. **Skew Towards Positive**: VADER predicts **73.15% positive** feedback, which is significantly higher than the rating-based ground truth of **55.22% positive** reviews. This positive bias is highly typical for B2B communications, where customers employ polite business phrases (e.g., "We appreciate", "please", "thank you") even when reporting operational or quality-related issues.
2. **Extremely Low Neutral Predictions**: VADER predicts only **3.90% neutral** reviews, compared to the **19.80% neutral** (severity rating 3) in our ground truth. Rule-based models like VADER struggle to classify moderate statements as neutral, as even minor positive/negative words can shift the compound score past the standard threshold of $\pm 0.05$.
3. **Negative Class Alignment**: The negative class matches relatively well in overall percentage (**22.95% VADER negative** vs. **24.98% rating-based negative**), but the cross-tabulation reveals that VADER classified 725 rating-based negative feedback items as positive. This indicates that while the aggregate volumes align, there are significant individual classification errors, which Person 2 will evaluate in detail in the next phase.
